In [22]:
import pandas as pd
import torch
from torch import nn, optim
import numpy as np
b_jet = pd.read_csv("/kaggle/input/jet-tagging-lhcb/bjet_train.csv")
c_jet = pd.read_csv("/kaggle/input/jet-tagging-lhcb/cjet_train.csv")
l_jet = pd.read_csv("/kaggle/input/jet-tagging-lhcb/ljet_train.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [23]:
cols = ['PT', 'ETA', 'drSvrJet', 'fdChi2', 'fdrMin', 'm', 'mCor',
       'mCorErr', 'pt', 'ptSvrJet', 'tau', 'ipChi2Sum'] 
# Excluded the existing quark flavour label "mc_flavour"
cols

['PT',
 'ETA',
 'drSvrJet',
 'fdChi2',
 'fdrMin',
 'm',
 'mCor',
 'mCorErr',
 'pt',
 'ptSvrJet',
 'tau',
 'ipChi2Sum']

In [24]:
b_jet["label"] = 0
c_jet["label"] = 1
l_jet["label"] = 2
# Select an equal amount of samples from each file to ensure equal exposure to each class
final_data = pd.concat((b_jet.iloc[:21000, :], c_jet.iloc[:21000, :], l_jet.iloc[:, :]), axis=0, ignore_index=True)

In [25]:
final_data.head()

,PT,ETA,mc_flavour,drSvrJet,fdChi2,fdrMin,m,mCor,mCorErr,nTrk,nTrkJet,pt,ptSvrJet,tau,ipChi2Sum,label
0,20946.177438,2.707440,5,0.035251,141.475691,0.542914,944.284764,1540.899314,151.282137,3.0,3.0,3945.870645,0.188381,0.156144,107.797163,0
1,29216.333273,2.556488,5,0.127721,170.004259,0.506962,1913.931833,3182.253839,297.165250,3.0,2.0,5603.219399,0.191784,0.093517,143.267777,0
2,22313.861629,2.973624,5,0.153069,394.557129,0.686985,1527.646610,2132.897943,138.362209,3.0,3.0,10787.480138,0.483443,0.106763,139.803156,0
3,22709.069322,4.138024,5,0.280194,1269.791481,2.481682,661.560641,1617.518090,69.681447,2.0,2.0,4665.633712,0.205452,0.626812,608.789159,0
4,61147.256182,2.354720,5,0.132560,2345.418371,9.111015,1208.325354,1606.143053,30.367378,3.0,3.0,12417.593659,0.203077,0.785546,14529.534651,0


## Data processing : Normalising range

In [26]:
for col in cols:
    print(final_data[col].mean())
    final_data[col] = final_data[col]/final_data[col].mean()

45870.53043030154
2.87833988164665
0.10114389211446896
42481.40468273734
2.5810137960408683
1812.3066303553483
3378.8750198216626
751.4169293555159
14500.724723824049
0.34062199387746256
0.2700994936903479
2784.1718717606655


In [27]:
print(final_data.head())

         PT       ETA  mc_flavour  drSvrJet    fdChi2    fdrMin         m  \
0  0.456637  0.940625           5  0.348526  0.003330  0.210349  0.521040   
1  0.636930  0.888181           5  1.262762  0.004002  0.196420  1.056075   
2  0.486453  1.033104           5  1.513379  0.009288  0.266169  0.842929   
3  0.495069  1.437643           5  2.770251  0.029891  0.961515  0.365038   
4  1.333040  0.818083           5  1.310610  0.055210  3.530014  0.666733   

       mCor   mCorErr  nTrk  nTrkJet        pt  ptSvrJet       tau  ipChi2Sum  \
0  0.456039  0.201329   3.0      3.0  0.272115  0.553051  0.578099   0.038718   
1  0.941809  0.395473   3.0      2.0  0.386410  0.563040  0.346230   0.051458   
2  0.631245  0.184135   3.0      3.0  0.743927  1.419295  0.395274   0.050214   
3  0.478715  0.092733   2.0      2.0  0.321752  0.603168  2.320672   0.218661   
4  0.475348  0.040413   3.0      3.0  0.856343  0.596194  2.908358   5.218620   

   label  
0      0  
1      0  
2      0  
3     

In [28]:
final_data = final_data.drop("mc_flavour", axis=1)

In [29]:
from torch.utils.data import random_split

class QuarkJetDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.data = df
    def __getitem__(self, idx):
        return torch.tensor([self.data[column][idx] for column in cols], dtype=torch.float) , torch.tensor(self.data["label"][idx])
    def __len__(self):
        return len(self.data)
dataset = QuarkJetDataset(final_data)
train_data, test_data = random_split(dataset, [0.8, 0.2])

In [33]:
class QuarkClassificationModel(nn.Module):
    def __init__(self, in_size, hidden_size, dropout=0.2):
        super(QuarkClassificationModel, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_size, hidden_size),
            nn.Dropout(dropout),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 3)
        )
    def forward(self, x):
        return self.fc(x)
model_1 = QuarkClassificationModel(12, 32).to(device)
model_1(dataset[0][0].to(device))

tensor([0.3733, 0.0733, 0.3463], grad_fn=<ViewBackward0>)

In [34]:
BATCH_SIZE=32
train_loader = torch.utils.data.DataLoader(train_data, shuffle=True, batch_size=BATCH_SIZE)
test_loader = torch.utils.data.DataLoader(test_data, shuffle=True, batch_size=BATCH_SIZE)
loss_fn = nn.CrossEntropyLoss()
optimiser = optim.Adam(model_1.parameters(), lr=5e-4)
EPOCHS = 5
def train(epochs, model, opt, loss_func):
    for epoch in range(epochs):
        epoch_loss = 0
        
        for i, batch in enumerate(train_loader):
            opt.zero_grad()
            output = model(batch[0].to(device)).to(device)
            loss = loss_func(output, batch[1].to(device))
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
            if i % 200 == 0:
                print(f"Batch {i} : Loss {loss.item()}")            
            
        print(f"Epoch : {epoch+1} | Loss {epoch_loss/len(train_loader)}")

        with torch.no_grad():
            num_correct = 0
            for i, batch in enumerate(test_loader):
                output = model(batch[0].to(device)).to(device)
                for x in range(len(output)-1):
                    num_correct += (torch.argmax(output[x]) == batch[1][x])
            print("Accuracy : ", num_correct/len(dataset))
train(EPOCHS, model_1, optimiser, loss_fn)

Batch 0 : Loss 1.2418161630630493
Batch 200 : Loss 0.9109839200973511
Batch 400 : Loss 0.7586265802383423
Batch 600 : Loss 0.6619508266448975
Batch 800 : Loss 0.7419959306716919
Batch 1000 : Loss 0.6262531280517578
Batch 1200 : Loss 0.772496223449707
Batch 1400 : Loss 0.6654168963432312
Epoch : 1 | Loss 0.7861815855886499
Accuracy :  tensor(0.1375)
Batch 0 : Loss 0.8617262840270996
Batch 200 : Loss 0.5250552296638489
Batch 400 : Loss 0.6412312984466553
Batch 600 : Loss 0.6715717315673828
Batch 800 : Loss 0.7193666100502014
Batch 1000 : Loss 0.6960424780845642
Batch 1200 : Loss 0.6404918432235718
Batch 1400 : Loss 0.7925714254379272
Epoch : 2 | Loss 0.7078369962575715
Accuracy :  tensor(0.1389)
Batch 0 : Loss 0.8152806162834167
Batch 200 : Loss 0.5619359612464905
Batch 400 : Loss 0.6394463777542114
Batch 600 : Loss 0.8285595774650574
Batch 800 : Loss 0.8651846647262573
Batch 1000 : Loss 0.5679361820220947
Batch 1200 : Loss 0.75083327293396
Batch 1400 : Loss 0.8240967988967896
Epoch : 3 

In [35]:
import sklearn
from sklearn.model_selection import train_test_split

In [36]:
from sklearn.ensemble import RandomForestClassifier

In [45]:
X = final_data.iloc[:, :14]
Y = final_data.iloc[:, 14]
Y
X_train, X_test, Y_train, Y_test = train_test_split(X, Y)

rf_classifier = RandomForestClassifier(n_estimators=128, criterion="gini", max_depth=16)
rf_classifier.fit(X_train, Y_train)
rf_classifier.score(X_test, Y_test)

0.8149850022337098